# 01 - Data Collection

**Goal:** load the raw IEC LGE 2016 and 2021 results files (one CSV per province, per year) into two combined DataFrames, and do a first pass of sanity checks before any cleaning happens.

**Source data:**
- 2016 LGE results - https://results.elections.org.za/home/downloads/me-results
- 2021 LGE results - https://results.elections.org.za/home/downloads/me-results

**Expected raw layout:**
```
data/raw/LGE2016/<PROVINCE_CODE>.csv   e.g. EC.csv, FS.csv, GT.csv ...
data/raw/LGE2021/<PROVINCE_CODE>.csv
```
Each file is at **voting-station × party** grain - one row per party per voting station, with station-level fields (`RegisteredVoters`, `SpoiltVotes`, `BallotType`) repeated across every party row for that station. That repetition is handled in `02_cleaning_and_merge`, not here.

In [82]:
import glob
import pandas as pd

pd.set_option("display.max_columns", None)

## Load all province files for each year

In [83]:
def load_province_files(year_dir: str,  encoding: str = "utf-8") -> pd.DataFrame:
    """Load and concatenate every per-province CSV in a year's raw data folder."""
    files = sorted(glob.glob(f"{year_dir}/*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {year_dir}")
    print(f"{year_dir}: found {len(files)} province files")
    return pd.concat(
        [pd.read_csv(f, encoding=encoding) for f in files], ignore_index=True
    )
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

In [84]:
df_2016 = load_province_files("../data/raw/LGE2016")
df_2021 = load_province_files("../data/raw/LGE2021")

print("2016 rows:", len(df_2016))
print("2021 rows:", len(df_2021))

../data/raw/LGE2016: found 9 province files
../data/raw/LGE2021: found 9 province files
2016 rows: 653063
2021 rows: 1084734


> Note: Raw row counts nearly doubled from 2016 to 2021 (653k → 1,084k rows). This was investigated and found to be explained by growth in contesting parties/candidates (206 → 324), not duplication - voting district counts stayed stable (22,612 → 23,147, ~2% growth).

In [85]:
def to_ward_level(df):
    # Filter to the Ward ballot only (PR and DC 40% are separate ballots that
    # would otherwise inflate vote totals if summed in alongside Ward votes)
    df = df[df["BallotType"] == "Ward"].copy()

    # Step 1: collapse to one row per voting station
    station = df.groupby(
        ["Province", "Municipality", "Ward", "VotingDistrict"], as_index=False
    ).agg(
        RegisteredVoters=("RegisteredVoters", "first"),
        SpoiltVotes=("SpoiltVotes", "first"),
        TotalValidVotes=("TotalValidVotes", "sum")
    )
    # Step 2: aggregate stations up to ward
    ward = station.groupby(
        ["Province", "Municipality", "Ward"], as_index=False
    ).agg(
        RegisteredVoters=("RegisteredVoters", "sum"),
        SpoiltVotes=("SpoiltVotes", "sum"),
        TotalValidVotes=("TotalValidVotes", "sum")
    )
    ward["VotesCast"] = ward["TotalValidVotes"] + ward["SpoiltVotes"]
    ward["Turnout"] = ward["VotesCast"] / ward["RegisteredVoters"]
    return ward

ward_2016 = to_ward_level(df_2016)
ward_2021 = to_ward_level(df_2021)

# NOW do the overlap check
overlap = set(ward_2016["Ward"]) & set(ward_2021["Ward"])
print(f"2016 wards: {ward_2016['Ward'].nunique()}, 2021 wards: {ward_2021['Ward'].nunique()}, overlap: {len(overlap)}")

2016 wards: 4392, 2021 wards: 4468, overlap: 4349


### Structure

In [86]:
df_2016.head()

,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
0,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN CHRISTIAN DEMOCRATIC PARTY,3,8/11/2016 4:13:42 PM
1,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN INDEPENDENT CONGRESS,19,8/11/2016 4:13:42 PM
2,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN NATIONAL CONGRESS,347,8/11/2016 4:13:42 PM
3,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,CONGRESS OF THE PEOPLE,7,8/11/2016 4:13:42 PM
4,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,DEMOCRATIC ALLIANCE,751,8/11/2016 4:13:42 PM


In [87]:
df_2016.info()

<class 'pandas.DataFrame'>
RangeIndex: 653063 entries, 0 to 653062
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Province           653063 non-null  str  
 1   Municipality       653063 non-null  str  
 2   Ward               653063 non-null  str  
 3   VotingDistrict     653063 non-null  int64
 4   VotingStationName  653063 non-null  str  
 5   RegisteredVoters   653063 non-null  int64
 6   BallotType         653063 non-null  str  
 7   SpoiltVotes        653063 non-null  int64
 8   PartyName          653063 non-null  str  
 9   TotalValidVotes    653063 non-null  int64
 10  DateGenerated      653063 non-null  str  
dtypes: int64(4), str(7)
memory usage: 127.3 MB


In [88]:
df_2021.head()

,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
0,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2724,PR,1,ABANTU BATHO CONGRESS,3,11/23/2021 4:17:50 PM
1,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2724,PR,1,AFRICA RESTORATION ALLIANCE,6,11/23/2021 4:17:50 PM
2,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2724,PR,1,AFRICAN CHRISTIAN DEMOCRATIC PARTY,3,11/23/2021 4:17:50 PM
3,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2724,PR,1,AFRICAN INDEPENDENT CONGRESS,7,11/23/2021 4:17:50 PM
4,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2724,PR,1,AFRICAN MULTICULTURAL ECONOMIC CONGRESS,0,11/23/2021 4:17:50 PM


In [89]:
df_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 1084734 entries, 0 to 1084733
Data columns (total 11 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   Province           1084734 non-null  str  
 1   Municipality       1084734 non-null  str  
 2   Ward               1084734 non-null  str  
 3   VotingDistrict     1084734 non-null  int64
 4   VotingStationName  1084564 non-null  str  
 5   RegisteredVoters   1084734 non-null  int64
 6   BallotType         1084734 non-null  str  
 7   SpoiltVotes        1084734 non-null  int64
 8   PartyName          1084734 non-null  str  
 9   TotalValidVotes    1084734 non-null  int64
 10  DateGenerated      1084734 non-null  str  
dtypes: int64(4), str(7)
memory usage: 212.1 MB


### Sanity checks

In [90]:
# Ballot types present - LGE ballots include Ward, PR, and (in local
# municipalities only) DC 40%. We deliberately keep all of them here;
# 02_cleaning_and_merge filters to a single ballot type before aggregating.
print("2016 BallotType values:", df_2016["BallotType"].unique())
print("2021 BallotType values:", df_2021["BallotType"].unique())


2016 BallotType values: <ArrowStringArray>
['PR', 'Ward', 'DC 40%']
Length: 3, dtype: str
2021 BallotType values: <ArrowStringArray>
['PR', 'Ward', 'DC 40%']
Length: 3, dtype: str


In [91]:
# Missing values check
print("2016 missing values:\n", df_2016.isna().sum())
print("\n2021 missing values:\n", df_2021.isna().sum())

2016 missing values:
 Province             0
Municipality         0
Ward                 0
VotingDistrict       0
VotingStationName    0
RegisteredVoters     0
BallotType           0
SpoiltVotes          0
PartyName            0
TotalValidVotes      0
DateGenerated        0
dtype: int64

2021 missing values:
 Province               0
Municipality           0
Ward                   0
VotingDistrict         0
VotingStationName    170
RegisteredVoters       0
BallotType             0
SpoiltVotes            0
PartyName              0
TotalValidVotes        0
DateGenerated          0
dtype: int64


> Note: VotingStationName was missing for 170 rows (2 voting districts) in the 2021 data. All other fields (VotingDistrict, RegisteredVoters, BallotType, vote counts) were complete for these rows. Since ward-level aggregation groups on VotingDistrict rather than station name, this has no effect on turnout calculations.

In [92]:
# Ward ID format check - both years should use the same 'Ward XXXXXXXX' format
print(df_2016["Ward"].sample(5, random_state=1).tolist())
print(df_2021["Ward"].sample(5, random_state=1).tolist())

['Ward 64005028', 'Ward 79800043', 'Ward 83206011', 'Ward 93404032', 'Ward 63803006']
['Ward 52402003', 'Ward 29300028', 'Ward 63803006', 'Ward 10202010', 'Ward 41601007']


In [93]:
# Unique ward counts per year
print("2016 unique wards (raw):", df_2016["Ward"].nunique())
print("2021 unique wards (raw):", df_2021["Ward"].nunique())

2016 unique wards (raw): 4392
2021 unique wards (raw): 4468


### Notes



In [94]:
df_2011 = load_province_files("../data/raw/LGE2011", encoding="utf-16")
print("2011 rows:", len(df_2011))

../data/raw/LGE2011: found 9 province files
2011 rows: 474386


In [95]:
pd.set_option("display.max_colwidth", None)
print(df_2011["Municipality"].iloc[0])
print(df_2011["Municipality"].nunique())
print(df_2011["BallotType"].unique())

ward_2011 = to_ward_level(df_2011)
overlap_2011_2016 = set(ward_2011["Ward"]) & set(ward_2016["Ward"])
print(f"2011 wards: {ward_2011['Ward'].nunique()}, "
      f"2016 wards: {ward_2016['Ward'].nunique()}, "
      f"overlap: {len(overlap_2011_2016)}")

BUF - Buffalo City Metropolitan Municipality [East London]
234
<ArrowStringArray>
['PR', 'Ward', 'DC 40%']
Length: 3, dtype: str
2011 wards: 4277, 2016 wards: 4392, overlap: 3836


In [96]:
import inspect
print(inspect.getsource(to_ward_level))

test_slice = df_2011[df_2011["Ward"] == "Ward 29200001"].copy()
print("Rows in slice, by BallotType:", test_slice["BallotType"].value_counts())

result = to_ward_level(test_slice)
print(result)

def to_ward_level(df):
    # Filter to the Ward ballot only (PR and DC 40% are separate ballots that
    # would otherwise inflate vote totals if summed in alongside Ward votes)
    df = df[df["BallotType"] == "Ward"].copy()

    # Step 1: collapse to one row per voting station
    station = df.groupby(
        ["Province", "Municipality", "Ward", "VotingDistrict"], as_index=False
    ).agg(
        RegisteredVoters=("RegisteredVoters", "first"),
        SpoiltVotes=("SpoiltVotes", "first"),
        TotalValidVotes=("TotalValidVotes", "sum")
    )
    # Step 2: aggregate stations up to ward
    ward = station.groupby(
        ["Province", "Municipality", "Ward"], as_index=False
    ).agg(
        RegisteredVoters=("RegisteredVoters", "sum"),
        SpoiltVotes=("SpoiltVotes", "sum"),
        TotalValidVotes=("TotalValidVotes", "sum")
    )
    ward["VotesCast"] = ward["TotalValidVotes"] + ward["SpoiltVotes"]
    ward["Turnout"] = ward["VotesCast"] / ward["RegisteredVoters"]
    return

In [97]:
print(df_2011["BallotType"].unique())
for v in df_2011["BallotType"].unique():
    print(repr(v), len(v))
print("Exact match count:", (df_2011["BallotType"] == "Ward").sum())
print("Stripped match count:", (df_2011["BallotType"].str.strip() == "Ward").sum())
print("Contains match count:", df_2011["BallotType"].str.contains("Ward", na=False).sum())

<ArrowStringArray>
['PR', 'Ward', 'DC 40%']
Length: 3, dtype: str
'PR' 2
'Ward' 4
'DC 40%' 6
Exact match count: 154132
Stripped match count: 154132
Contains match count: 154132


In [98]:
# after a full restart and run-all
print(ward_2011[ward_2011["Ward"] == "Ward 29200001"])
# Sanity check: recompute one known ward directly from raw data and compare
_raw = df_2011[(df_2011["Ward"] == "Ward 29200001") & (df_2011["BallotType"] == "Ward")]
_expected_votes = _raw["TotalValidVotes"].sum() + _raw.groupby("VotingDistrict")["SpoiltVotes"].first().sum()
_actual_votes = ward_2011.loc[ward_2011["Ward"] == "Ward 29200001", "VotesCast"].iloc[0]
assert abs(_expected_votes - _actual_votes) < 1, f"Mismatch: expected {_expected_votes}, got {_actual_votes}"
print("Sanity check passed.")

       Province                                                Municipality  \
0  Eastern Cape  BUF - Buffalo City Metropolitan Municipality [East London]   

            Ward  RegisteredVoters  SpoiltVotes  TotalValidVotes  VotesCast  \
0  Ward 29200001              6708           83             3616       3699   

    Turnout  
0  0.551431  
Sanity check passed.


In [99]:
missing_from_2016 = set(ward_2011["Ward"]) - set(ward_2016["Ward"])
print(ward_2011[ward_2011["Ward"].isin(missing_from_2016)][["Province", "Municipality", "Ward"]].head(10))

panel_11_16 = ward_2011.merge(
    ward_2016, on=["Province", "Ward"], suffixes=("_2011", "_2016")
)
print("2011+2016 merged:", panel_11_16.shape)

ward_panel_3yr = panel_11_16.merge(
    ward_2021, on=["Province", "Ward"], suffixes=("", "_2021")
)
print("Final 3-year panel:", ward_panel_3yr.shape)

outliers_2011 = ward_panel_3yr[
    (ward_panel_3yr["Turnout_2011"] > 1.0) | (ward_panel_3yr["Turnout_2011"] < 0.05)
]
print(len(outliers_2011))
outliers_2011[["Province", "Municipality_2011", "Ward", "RegisteredVoters_2011", "VotesCast_2011", "Turnout_2011"]]

         Province                     Municipality           Ward
63   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003001
64   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003002
65   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003003
66   Eastern Cape     EC103 - Ikwezi [Jansenville]  Ward 21003004
99   Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007001
100  Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007002
101  Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007003
102  Eastern Cape    EC107 - Baviaans [Willowmore]  Ward 21007004
208  Eastern Cape  EC124 - Amahlathi [Stutterheim]  Ward 21204016
209  Eastern Cape  EC124 - Amahlathi [Stutterheim]  Ward 21204017
2011+2016 merged: (3836, 14)
Final 3-year panel: (3834, 20)
0


,Province,Municipality_2011,Ward,RegisteredVoters_2011,VotesCast_2011,Turnout_2011


In [100]:
# For a station-level table filtered to Ward ballot only, does RegisteredVoters
# vary within a single VotingDistrict? It shouldn't -- if it does, VotingDistrict
# isn't a clean station identifier in 2011.
df_2011_ward = df_2011[df_2011["BallotType"] == "Ward"]

check = df_2011_ward.groupby(["Province", "Ward", "VotingDistrict"])["RegisteredVoters"].nunique()
print("VotingDistricts with inconsistent RegisteredVoters:", (check > 1).sum())
print("out of total VotingDistricts:", len(check))

# Also check whether multiple distinct station names share one VotingDistrict code
station_names_per_district = df_2011_ward.groupby("VotingDistrict")["VotingStationName"].nunique()
print("VotingDistricts with >1 station name:", (station_names_per_district > 1).sum())

dupes = df_2011_ward.duplicated(subset=["VotingDistrict", "PartyName"], keep=False)
print("Duplicate party-station rows in 2011:", dupes.sum())

VotingDistricts with inconsistent RegisteredVoters: 0
out of total VotingDistricts: 20857
VotingDistricts with >1 station name: 0
Duplicate party-station rows in 2011: 0


In [101]:
def votes_for_ballot(df, ballot_type):
    d = df[df["BallotType"] == ballot_type]
    station = d.groupby(["Province", "Ward", "VotingDistrict"], as_index=False).agg(
        RegisteredVoters=("RegisteredVoters", "first"),
        TotalValidVotes=("TotalValidVotes", "sum"),
    )
    ward = station.groupby(["Province", "Ward"], as_index=False).agg(
        RegisteredVoters=("RegisteredVoters", "sum"),
        TotalValidVotes=("TotalValidVotes", "sum"),
    )
    ward["Turnout"] = ward["TotalValidVotes"] / ward["RegisteredVoters"]
    return ward

pr_check = votes_for_ballot(df_2011, "PR")
print(pr_check[pr_check["Ward"] == "Ward 29200001"])

rows_2011 = df_2011[df_2011["BallotType"]=="Ward"].groupby("VotingDistrict").size()
rows_2016 = df_2016[df_2016["BallotType"]=="Ward"].groupby("VotingDistrict").size()
print("2011 avg Ward-ballot rows per station:", rows_2011.mean())
print("2016 avg Ward-ballot rows per station:", rows_2016.mean())

print("2011 only:", set(df_2011.columns) - set(df_2016.columns))
print("2016 only:", set(df_2016.columns) - set(df_2011.columns))

         Province           Ward  RegisteredVoters  TotalValidVotes   Turnout
605  Eastern Cape  Ward 29200001              6708             3647  0.543679
2011 avg Ward-ballot rows per station: 7.389941026993336
2016 avg Ward-ballot rows per station: 8.69998231027773
2011 only: set()
2016 only: set()


In [102]:
raw_ward_ballot = df_2011[
    (df_2011["VotingDistrict"] == 10590151) & (df_2011["BallotType"] == "Ward")
]
print(raw_ward_ballot[["PartyName", "TotalValidVotes", "DateGenerated"]])

                                     PartyName  TotalValidVotes  \
8           AFRICAN CHRISTIAN DEMOCRATIC PARTY                5   
9                    AFRICAN NATIONAL CONGRESS              444   
10                     CONGRESS  OF THE PEOPLE               32   
11  DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE              678   
12           PAN AFRICANIST CONGRESS OF AZANIA                8   

           DateGenerated  
8   5/26/2011 4:36:55 PM  
9   5/26/2011 4:36:55 PM  
10  5/26/2011 4:36:55 PM  
11  5/26/2011 4:36:55 PM  
12  5/26/2011 4:36:55 PM  


In [103]:
print(f"ward_2011 rows: {len(ward_2011)}, unique Ward values: {ward_2011['Ward'].nunique()}")
raw_full_ward = df_2011[(df_2011["Ward"] == "Ward 29200001") & (df_2011["BallotType"] == "Ward")]
station_totals = raw_full_ward.groupby(["VotingDistrict", "VotingStationName"]).agg(
    RegisteredVoters=("RegisteredVoters", "first"),
    TotalValidVotes=("TotalValidVotes", "sum"),
)
print(station_totals)
print(station_totals.sum())

ward_2011 rows: 4277, unique Ward values: 4277
                                                       RegisteredVoters  \
VotingDistrict VotingStationName                                          
10590151       PEFFERVILLE CLINIC                                  2421   
10590858       SHAD MASHOLOGU MEMORIAL BAPTIST CHURCH              2459   
10590869       MASAKHE PUBLIC SCHOOL                               1563   
10591297       BRAELYN COMMUNITY HALL                               265   

                                                       TotalValidVotes  
VotingDistrict VotingStationName                                        
10590151       PEFFERVILLE CLINIC                                 1167  
10590858       SHAD MASHOLOGU MEMORIAL BAPTIST CHURCH             1434  
10590869       MASAKHE PUBLIC SCHOOL                               864  
10591297       BRAELYN COMMUNITY HALL                              151  
RegisteredVoters    6708
TotalValidVotes     3616
dtype: int64


In [104]:
df_2011_ward = df_2011[df_2011["BallotType"] == "Ward"]
spoilt_check = df_2011_ward.groupby(["Province", "Ward", "VotingDistrict"])["SpoiltVotes"].nunique()
print("VotingDistricts with inconsistent SpoiltVotes:", (spoilt_check > 1).sum())
print("out of total VotingDistricts:", len(spoilt_check))

raw_full_ward = df_2011[(df_2011["Ward"] == "Ward 29200001") & (df_2011["BallotType"] == "Ward")]
station_totals = raw_full_ward.groupby(["VotingDistrict", "VotingStationName"]).agg(
    RegisteredVoters=("RegisteredVoters", "first"),
    SpoiltVotes=("SpoiltVotes", "first"),
    TotalValidVotes=("TotalValidVotes", "sum"),
)
print(station_totals)
print(station_totals.sum())

VotingDistricts with inconsistent SpoiltVotes: 0
out of total VotingDistricts: 20857
                                                       RegisteredVoters  \
VotingDistrict VotingStationName                                          
10590151       PEFFERVILLE CLINIC                                  2421   
10590858       SHAD MASHOLOGU MEMORIAL BAPTIST CHURCH              2459   
10590869       MASAKHE PUBLIC SCHOOL                               1563   
10591297       BRAELYN COMMUNITY HALL                               265   

                                                       SpoiltVotes  \
VotingDistrict VotingStationName                                     
10590151       PEFFERVILLE CLINIC                               23   
10590858       SHAD MASHOLOGU MEMORIAL BAPTIST CHURCH           46   
10590869       MASAKHE PUBLIC SCHOOL                            14   
10591297       BRAELYN COMMUNITY HALL                            0   

                                           